# Sistem Rekomendasi Buku (Book Recommendation System)
**Mata Kuliah: Analisis Data Lanjut (Tingkat Magister / S2)**  
*Eksperimen Komprehensif Algoritma Rekomendasi pada Dataset Book-Crossing*

---

## Ringkasan Proyek
Notebook ini merupakan dokumen eksperimen terstruktur yang mendasari pembuatan sistem rekomendasi interaktif. Di sini dibahas secara mendalam:
1. **Pembersihan Data & Audit Kualitas:** Penanganan *missing values*, anomali demografi, dan pemisahan rating eksplisit vs implisit.
2. **Karakteristik Matriks:** Analisis *extreme sparsity* ($>99.9\%$), *positivity bias*, dan fenomena *long-tail*.
3. **Representasi Fitur & Normalisasi:** *k-core pruning*, *Mean-Centering*, *Bayesian Shrinkage*, ekstraksi *TF-IDF metadata soup*, dan dekomposisi *TruncatedSVD*.
4. **Implementasi 4 Paradigma Rekomendasi:**
   - **Rule-Based:** *Bayesian Weighted Rating (IMDb)* & *Association Rules Mining*.
   - **Content-Based Filtering:** *TF-IDF vector space* + *Cosine similarity* + *XAI (Explainability)*.
   - **Collaborative Filtering:** *Item-Based CF*, *User-Based CF (k-NN)*, dan *SVD Matrix Factorization*.
   - **Hybrid Systems:** *Weighted Hybrid* dan *Switching Hybrid* (solusi *cold-start*).
5. **Evaluasi Offline Terstandarisasi:** Skema *hold-out validation* per pengguna untuk mengukur akurasi prediksi (*RMSE, MAE*) dan kualitas perankingan (*Precision@K, Recall@K, HitRate@K*).


---
## Bagian 1: Setup Lingkungan & Ingestion Data

Notebook ini dirancang agar dapat dijalankan secara instan di **Google Colab** maupun di **lingkungan lokal**.
Jika dijalankan di Google Colab, kode di bawah akan secara otomatis mengkloning dataset dari repositori GitHub:
`https://github.com/akbaarrmaulana/Book-Recommendation-adl`.


In [ ]:
import os
import sys
import html
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from scipy.stats import rankdata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 100

# Deteksi apakah berjalan di Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Lingkungan: Google Colab terdeteksi.")
    REPO_URL = "https://github.com/akbaarrmaulana/Book-Recommendation-adl.git"
    REPO_DIR = "Book-Recommendation-adl"
    if not os.path.exists(REPO_DIR):
        print(f"Mengunduh dataset dari {REPO_URL}...")
        !git clone {REPO_URL}
    DATA_DIR = os.path.join(REPO_DIR, "data")
else:
    print("Lingkungan: Komputer Lokal terdeteksi.")
    DATA_DIR = "data" if os.path.exists("data") else os.path.join("bookrec", "data")

print(f"Direktori data aktif: {os.path.abspath(DATA_DIR)}")
assert os.path.exists(DATA_DIR), f"Folder data tidak ditemukan di {DATA_DIR}!"


---
## Bagian 2: Data Ingestion & Cleansing

Dataset *Book-Crossing* terdiri dari 3 tabel relasional:
1. `BX-Users.csv`: Profil pengguna (`User-ID`, `Location`, `Age`).
2. `BX-Books.csv`: Katalog buku (`ISBN`, `Book-Title`, `Book-Author`, `Year-Of-Publication`, `Publisher`, `Image-URL-M`).
3. `BX-Book-Ratings.csv`: Catatan interaksi pengguna terhadap buku (`User-ID`, `ISBN`, `Book-Rating`).

### Kebijakan Pembersihan Data:
- **Pemisahan Rating Implisit vs Eksplisit:** Nilai `0` pada dataset ini melambangkan interaksi implisit (misal: pengguna membaca/menyimpan buku tanpa memberi penilaian). Nilai ini wajib dipisahkan dari rating eksplisit ($1	ext{–}10$) agar skala ordinal prediksi rating tidak bias.
- **Anomali Usia:** Rentang usia dibatasi pada $5 \le 	ext{Age} \le 100$. Nilai di luar rentang ini dikonversi menjadi `NaN` dan diklasifikasikan ke dalam kelompok *"Tidak diketahui"*.
- **Parsing Geografis:** Kolom `Location` ('City, State, Country') diurai untuk mengambil entri negara.
- **Tahun Terbit:** Dibatasi antara $1800 \le 	ext{Year} \le 2006$ (dataset dikumpulkan pada 2004).


In [ ]:
AGE_BINS = [5, 18, 25, 35, 45, 55, 65, 101]
AGE_LABELS = ["<18", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
AGE_UNKNOWN = "Tidak diketahui"
AGE_ORDER = AGE_LABELS + [AGE_UNKNOWN]
MAX_YEAR = 2006

def clean_users(df_users: pd.DataFrame) -> pd.DataFrame:
    u = df_users.rename(columns={"User-ID": "user_id"}).copy()
    u["Age"] = pd.to_numeric(u["Age"], errors="coerce")
    u.loc[(u["Age"] < 5) | (u["Age"] > 100), "Age"] = np.nan
    
    parts = u["Location"].fillna("").astype(str).str.split(",")
    u["City"] = parts.str[0].str.strip().str.title()
    country = parts.str[-1].str.strip().str.title()
    country = country.replace({
        "Usa": "USA", "Us": "USA", "U.S.A.": "USA", "United States": "USA",
        "Uk": "United Kingdom", "": "Unknown", "N/A": "Unknown"
    })
    u["Country"] = country
    
    ag = pd.cut(u["Age"], bins=AGE_BINS, labels=AGE_LABELS, right=False).astype("object")
    u["AgeGroup"] = ag.where(ag.notna(), AGE_UNKNOWN)
    return u

def clean_books(df_books: pd.DataFrame) -> pd.DataFrame:
    b = df_books.rename(columns={
        "ISBN": "isbn", "Book-Title": "title", "Book-Author": "author",
        "Year-Of-Publication": "year", "Publisher": "publisher",
        "Image-URL-M": "image"
    }).copy()
    b["isbn"] = b["isbn"].astype(str).str.strip()
    for col in ["title", "author", "publisher"]:
        b[col] = b[col].fillna("Unknown").astype(str).map(html.unescape).str.strip()
    b["year"] = pd.to_numeric(b["year"], errors="coerce")
    b.loc[(b["year"] < 1800) | (b["year"] > MAX_YEAR), "year"] = np.nan
    b["decade"] = (b["year"] // 10) * 10
    return b.drop_duplicates("isbn")

# Baca file mentah
print("Membaca file dataset mentah...")
raw_ratings = pd.read_csv(os.path.join(DATA_DIR, "BX-Book-Ratings.csv"), sep=";", encoding="latin-1", on_bad_lines="skip")
raw_books = pd.read_csv(os.path.join(DATA_DIR, "BX-Books.csv"), sep=";", encoding="latin-1", escapechar="\\",
                        usecols=["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher", "Image-URL-M"], on_bad_lines="skip")
raw_users = pd.read_csv(os.path.join(DATA_DIR, "BX-Users.csv"), sep=";", encoding="latin-1", on_bad_lines="skip")

# Bersihkan masing-masing tabel
users = clean_users(raw_users)
books = clean_books(raw_books)

ratings = raw_ratings.rename(columns={"User-ID": "user_id", "ISBN": "isbn", "Book-Rating": "rating"})
ratings["isbn"] = ratings["isbn"].astype(str).str.strip()
ratings = ratings.drop_duplicates(["user_id", "isbn"], keep="last")

# Pisahkan rating eksplisit (1-10)
explicit = ratings[ratings["rating"] > 0].copy()
explicit = explicit[explicit["isbn"].isin(books["isbn"])]
explicit = explicit.merge(users[["user_id", "Age", "AgeGroup", "Country"]], on="user_id", how="left")
explicit["AgeGroup"] = explicit["AgeGroup"].fillna(AGE_UNKNOWN)
explicit["Country"] = explicit["Country"].fillna("Unknown")

print(f"Total Pengguna Terdaftar     : {len(users):,}")
print(f"Total Judul Buku di Katalog  : {len(books):,}")
print(f"Total Interaksi Rating       : {len(ratings):,}")
print(f"Interaksi Rating Eksplisit   : {len(explicit):,} ({100*len(explicit)/len(ratings):.1f}%)")
print(f"Interaksi Rating Implisit (0): {len(ratings)-len(explicit):,} ({100*(len(ratings)-len(explicit))/len(ratings):.1f}%)")


---
## Bagian 3: Analisis Eksplorasi (EDA) & Kepadatan Matriks (*Sparsity*)

Tantangan fundamental dalam sistem rekomendasi buku adalah **Extreme Sparsity**.
Kepadatan matriks interaksi dihitung dengan:
$$\text{Density} = \frac{|R|}{|U| \times |I|} \times 100\%$$
di mana $|R|$ adalah jumlah rating yang teramati, $|U|$ jumlah pengguna unik, dan $|I|$ jumlah buku unik.


In [ ]:
n_users_ex = explicit["user_id"].nunique()
n_books_ex = explicit["isbn"].nunique()
matrix_density = (len(explicit) / (n_users_ex * n_books_ex)) * 100

print(f"Pengguna unik (eksplisit): {n_users_ex:,}")
print(f"Buku unik (eksplisit)    : {n_books_ex:,}")
print(f"Kepadatan Matriks        : {matrix_density:.5f}% (Tingkat Sparsity: {100 - matrix_density:.5f}%)")

# Visualisasi EDA
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribusi Rating Eksplisit vs Implisit
rating_dist = ratings["rating"].value_counts().sort_index()
colors = ["#9CA3AF" if r == 0 else "#164E33" for r in rating_dist.index]
axes[0].bar(rating_dist.index, rating_dist.values, color=colors, edgecolor="none")
axes[0].set_title("Distribusi Nilai Rating (0 = Implisit, 1-10 = Eksplisit)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Nilai Rating")
axes[0].set_ylabel("Frekuensi")

# 2. Distribusi Frekuensi Rating per Pengguna (Cold-Start Problem)
user_counts = explicit.groupby("user_id").size()
axes[1].hist(user_counts.clip(upper=50), bins=30, color="#164E33", edgecolor="white")
axes[1].set_yscale("log")
axes[1].set_title("Frekuensi Rating per Pengguna (Sumbu-Y Log)", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Jumlah Rating yang Diberikan")
axes[1].set_ylabel("Jumlah Pengguna (Log)")

# 3. Kurva Long-Tail Popularitas Buku
book_counts = np.sort(explicit.groupby("isbn").size().values)[::-1]
cum_book_counts = np.cumsum(book_counts) / book_counts.sum() * 100
percent_books = np.linspace(0, 100, len(cum_book_counts))
axes[2].plot(percent_books, cum_book_counts, color="#164E33", lw=2)
axes[2].axhline(80, color="#E11D48", linestyle="--", label="80% Total Rating")
axes[2].set_title("Kurva Long-Tail Popularitas Buku", fontsize=11, fontweight="bold")
axes[2].set_xlabel("Persentase Judul Buku (%)")
axes[2].set_ylabel("Persentase Kumulatif Rating (%)")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Proporsi pengguna yang HANYA memberi 1 rating: {(user_counts == 1).mean() * 100:.2f}% (Tantangan Cold-Start)")


---
## Bagian 4: Pembentukan Matriks, Normalisasi & Reduksi Dimensi

### 1. K-Core Filtering
Untuk mereduksi *extreme sparsity*, dilakukan eliminasi iteratif (*k-core pruning*) sehingga setiap pengguna yang masuk model memiliki minimal $	ext{min\_user}$ rating, dan setiap buku memiliki minimal $	ext{min\_item}$ rating.

### 2. Mean-Centering & Bayesian Shrinkage
Pengguna memiliki skala kepuasan subjektif (*positivity bias*). Untuk menghilangkannya, rating dipusatkan terhadap rata-rata pengguna $\mu_u$.
Agar pengguna baru dengan 1–2 rating tidak menghasilkan estimasi yang tidak stabil, diterapkan **Bayesian Shrinkage** menuju rata-rata global $\mu_g$:
$$\mu_u = \frac{\sum_{i} r_{u,i} + \lambda_{\mu} \cdot \mu_g}{N_u + \lambda_{\mu}}$$
$$R_c(u, i) = R(u, i) - \mu_u$$

### 3. Reduksi Dimensi TruncatedSVD
Matriks rating terpusat $R_c$ didekomposisi menjadi representasi faktor laten:
$$R_c \approx U \Sigma V^T$$
di mana $V \in \mathbb{R}^{M \times k}$ adalah representasi laten buku pada ruang $k=50$ dimensi.

### 4. TF-IDF Metadata Soup
Informasi judul, penulis ($2\times$ bobot), penerbit, dan dekade digabungkan dan ditransformasikan menjadi representasi vektor teks menggunakan `TfidfVectorizer(sublinear_tf=True)`.


In [ ]:
SHRINK_MEAN = 2.0
LIKE_THR = 7  # Rating >= 7 diklasifikasikan sebagai 'disukai' (liked)

def filter_k_core(df: pd.DataFrame, min_user: int = 5, min_item: int = 5) -> pd.DataFrame:
    d = df.copy()
    for _ in range(6):
        n0 = len(d)
        d = d[d.groupby("user_id")["isbn"].transform("size") >= min_user]
        d = d[d.groupby("isbn")["user_id"].transform("size") >= min_item]
        if len(d) == n0:
            break
    return d

# Terapkan k-core pruning
clean_pairs = filter_k_core(explicit[["user_id", "isbn", "rating"]], min_user=5, min_item=5)
print(f"Data setelah k-core (min_user=5, min_item=5): {len(clean_pairs):,} pasangan rating")

# Bangun indeks pemetaan
user_list = np.sort(clean_pairs["user_id"].unique())
isbn_list = np.sort(clean_pairs["isbn"].unique())
user_idx_map = pd.Index(user_list)
isbn_idx_map = pd.Index(isbn_list)

rows = user_idx_map.get_indexer(clean_pairs["user_id"])
cols = isbn_idx_map.get_indexer(clean_pairs["isbn"])

# Matriks rating mentah R (Sparse CSR)
R = sparse.csr_matrix((clean_pairs["rating"].values.astype(np.float32), (rows, cols)),
                      shape=(len(user_list), len(isbn_list)))

# Hitung user_mean dengan Bayesian Shrinkage
global_mean = float(R.data.mean())
count_u = np.diff(R.indptr)
user_mean = (np.asarray(R.sum(1)).ravel() + SHRINK_MEAN * global_mean) / (count_u + SHRINK_MEAN)
user_mean = user_mean.astype(np.float32)

# Matriks rating terpusat Rc
coo = R.tocoo()
Rc = sparse.csr_matrix((coo.data - user_mean[coo.row], (coo.row, coo.col)), shape=R.shape)

# Matriks ternormalisasi L2 (untuk kemiripan kosinus cepat)
Xu = normalize(Rc)
Xi = normalize(Rc.T.tocsr())
Bu = R.copy()
Bu.data = np.ones_like(Bu.data)
Bi = Bu.T.tocsr()

# TruncatedSVD untuk model laten
k_factors = 50
svd = TruncatedSVD(n_components=k_factors, random_state=42).fit(Rc)
Vt = svd.components_.astype(np.float32)   # (k x items)
V = Vt.T.copy()                            # (items x k)

# Metadata items dalam indeks model
item_meta = books.set_index("isbn").reindex(isbn_list).reset_index()
n_ratings_per_item = np.asarray(Bu.sum(0)).ravel()
item_meta["n_ratings"] = n_ratings_per_item.astype(int)
item_meta["avg_rating"] = np.where(n_ratings_per_item > 0, np.asarray(R.sum(0)).ravel() / np.maximum(n_ratings_per_item, 1), np.nan)

# IMDb Weighted Rating per item
m_votes = max(5.0, float(np.quantile(n_ratings_per_item[n_ratings_per_item > 0], 0.75)))
item_meta["wr"] = (n_ratings_per_item / (n_ratings_per_item + m_votes)) * item_meta["avg_rating"].fillna(global_mean) + (m_votes / (n_ratings_per_item + m_votes)) * global_mean

# TF-IDF Metadata Soup
def make_soup(df: pd.DataFrame) -> pd.Series:
    tok = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
    au = "au_" + df["author"].map(tok)
    pb = "pb_" + df["publisher"].map(tok)
    dc = "dc_" + df["decade"].fillna(0).astype(int).astype(str)
    return df["title"].str.lower() + " " + au + " " + au + " " + pb + " " + dc

tfidf_vec = TfidfVectorizer(stop_words="english", min_df=2, sublinear_tf=True)
tfidf_mat = tfidf_vec.fit_transform(make_soup(item_meta)).astype(np.float32).tocsr()

# Matriks biner 'disukai' (Rating >= 7) untuk Association Rules
L = R.copy()
L.data = (L.data >= LIKE_THR).astype(np.float32)
L.eliminate_zeros()
L_csc = L.tocsc()
item_liked = np.asarray(L.sum(0)).ravel()

print(f"Matriks R Berhasil Dibentuk: {R.shape[0]:,} pengguna × {R.shape[1]:,} buku")
print(f"Dimensi Vektor Laten SVD  : {V.shape}")
print(f"Dimensi Matriks TF-IDF    : {tfidf_mat.shape}")


---
## Bagian 5: Implementasi Algoritma Rekomendasi

### 5.1 Rule-Based Recommenders
1. **Domain/Demographic Rules:** Rekomendasi buku terpopuler berdasarkan formula IMDb *Bayesian Weighted Rating*:
   $$WR = \frac{v}{v + m} R + \frac{m}{v + m} C$$
2. **Data-Driven Association Rules:** Menemukan pola transaksi buku yang disukai ($r \ge 7$):
   $$\text{Support}(A \rightarrow B) = \frac{|U_A \cap U_B|}{N}, \quad \text{Confidence}(A \rightarrow B) = \frac{|U_A \cap U_B|}{|U_A|}, \quad \text{Lift}(A \rightarrow B) = \frac{\text{Confidence}}{P(B)}$$


In [ ]:
def get_popular_books(country=None, age_group=None, min_votes=5, topn=5):
    d = explicit
    if country and country != "Semua":
        d = d[d["Country"] == country]
    if age_group and age_group != "Semua":
        d = d[d["AgeGroup"] == age_group]
    if d.empty:
        return pd.DataFrame()
    
    C = d["rating"].mean()
    g = d.groupby("isbn")["rating"].agg(n_ratings="count", avg_rating="mean").reset_index()
    g = g[g["n_ratings"] >= min_votes]
    if g.empty:
        return pd.DataFrame()
    
    g["score"] = (g["n_ratings"] / (g["n_ratings"] + min_votes)) * g["avg_rating"] + (min_votes / (g["n_ratings"] + min_votes)) * C
    res = g.merge(books[["isbn", "title", "author", "year"]], on="isbn").sort_values("score", ascending=False).head(topn)
    return res[["title", "author", "year", "n_ratings", "avg_rating", "score"]]

print("--- Contoh Rule-Based Demografis: Pengguna USA, Usia 25-34 ---")
display(get_popular_books(country="USA", age_group="25-34", topn=5))

def get_association_rules(basket_isbn_list, min_count=3, topn=5):
    basket_idx = [isbn_idx_map.get_loc(isbn) for isbn in basket_isbn_list if isbn in isbn_idx_map]
    if not basket_idx:
        return pd.DataFrame()
    
    N = L.shape[0]
    has_all = np.asarray(L_csc[:, basket_idx].sum(1)).ravel() == len(basket_idx)
    target_users = np.where(has_all)[0]
    n_basket = len(target_users)
    if n_basket == 0:
        return pd.DataFrame()
    
    co = np.asarray(L[target_users].sum(0)).ravel()
    for idx in basket_idx:
        co[idx] = 0
    
    keep = np.where(co >= min_count)[0]
    if len(keep) == 0:
        return pd.DataFrame()
    
    conf = co[keep] / n_basket
    supp = co[keep] / N
    p_b = item_liked[keep] / N
    lift = np.where(p_b > 0, conf / p_b, 0.0)
    
    out = item_meta.iloc[keep][["isbn", "title", "author", "year"]].copy()
    out["co_users"] = co[keep].astype(int)
    out["support"] = supp
    out["confidence"] = conf
    out["lift"] = lift
    return out.sort_values("lift", ascending=False).head(topn)

# Uji coba aturan asosiasi: Buku 'The Fellowship of the Ring'
sample_isbn = item_meta[item_meta['title'].str.contains('Fellowship of the Ring', case=False, na=False)]['isbn'].iloc[0]
print(f"\n--- Contoh Aturan Asosiasi untuk Buku: {item_meta.loc[item_meta['isbn']==sample_isbn, 'title'].iloc[0]} ---")
display(get_association_rules([sample_isbn], min_count=3, topn=5)[["title", "author", "confidence", "lift"]])


### 5.2 Content-Based Filtering
Membangun vektor profil pengguna dari rata-rata tertimbang representasi TF-IDF buku yang pernah disukai, lalu menghitung *Cosine Similarity* terhadap seluruh katalog buku.


In [ ]:
def predict_content_based(user_ratings_dict, topn=5):
    pairs = [(isbn_idx_map.get_loc(k), float(v)) for k, v in user_ratings_dict.items() if k in isbn_idx_map]
    if not pairs:
        return pd.DataFrame()
    idx, r = zip(*pairs)
    idx, r = np.array(idx), np.array(r, dtype=np.float32)
    
    # Ambil subset buku yang disukai (rating >= 7)
    liked_mask = r >= LIKE_THR
    if not liked_mask.any():
        liked_mask = r >= r.mean()
    li, lr = idx[liked_mask], r[liked_mask]
    
    # Bangun vektor profil pengguna
    user_prof = sparse.csr_matrix(lr.reshape(1, -1)) @ tfidf_mat[li]
    user_prof = normalize(user_prof)
    
    # Hitung Cosine Similarity ke semua buku
    scores = (tfidf_mat @ user_prof.T).toarray().ravel()
    scores[idx] = -np.inf  # Jangan rekomendasikan buku yang sudah dibaca
    
    best_idx = np.argsort(-scores)[:topn]
    res = item_meta.iloc[best_idx][["title", "author", "year"]].copy()
    res["score"] = scores[best_idx]
    
    # Fitur Penjelasan (Explainability / XAI)
    sim_to_liked = (tfidf_mat[best_idx] @ tfidf_mat[li].T).toarray()
    closest_liked = sim_to_liked.argmax(1)
    res["Alasan Rekomendasi"] = [f"Mirip: '{item_meta.iloc[li[c]]['title'][:35]}' (cos: {sim_to_liked[row, c]:.2f})"
                                  for row, c in enumerate(closest_liked)]
    return res

# Simulasi profil pembaca novel fantasi
sample_profile = {
    sample_isbn: 10.0
}
print(f"--- Rekomendasi Content-Based ---")
display(predict_content_based(sample_profile, topn=5))


### 5.3 Collaborative Filtering (CF)
1. **Item-Based CF (Memory-Based):** Kemiripan antar-item dihitung menggunakan *Adjusted Cosine* dengan peredam *Bayesian Shrinkage*:
   $$S'_{i,j} = S_{i,j} \cdot \frac{C_{i,j}}{C_{i,j} + \text{shrink}}$$
2. **User-Based CF (Memory-Based):** Mencari tetangga pengguna terdekat ($k=50$) pada ruang rating terpusat.
3. **SVD Matrix Factorization (Model-Based):** Memproyeksikan riwayat pengguna ke ruang faktor laten (*fold-in projection*) $\vec{z}_u = \vec{r}_u^c V$, lalu merekonstruksi estimasi rating $\hat{r} = \mu_u + \vec{z}_u V^T$.


In [ ]:
def predict_svd(user_ratings_dict, topn=5):
    pairs = [(isbn_idx_map.get_loc(k), float(v)) for k, v in user_ratings_dict.items() if k in isbn_idx_map]
    if not pairs:
        return pd.DataFrame()
    idx, r = zip(*pairs)
    idx, r = np.array(idx), np.array(r, dtype=np.float32)
    
    # Hitung mean pengguna dengan shrinkage
    mu_u = float((r.sum() + SHRINK_MEAN * global_mean) / (len(r) + SHRINK_MEAN))
    
    # Fold-in: Proyeksikan vektor residual pengguna ke ruang laten
    v_vec = sparse.csr_matrix((r - mu_u, (np.zeros(len(idx), dtype=int), idx)), shape=(1, len(isbn_list)))
    z = v_vec @ V  # (1 x k)
    
    # Rekonstruksi rating untuk seluruh katalog
    pred_ratings = mu_u + np.asarray(z @ Vt).ravel()
    pred_ratings[idx] = -np.inf
    
    best_idx = np.argsort(-pred_ratings)[:topn]
    res = item_meta.iloc[best_idx][["title", "author", "year"]].copy()
    res["pred_rating"] = pred_ratings[best_idx]
    return res

print("--- Rekomendasi Collaborative Filtering (SVD) ---")
display(predict_svd(sample_profile, topn=5))


### 5.4 Hybrid Recommender Systems
- **Weighted Hybrid:** Menghitung skor persentil terstandarisasi antara Content-Based dan CF, lalu memadukannya dengan bobot $\alpha$:
  $$\text{Skor} = \alpha \cdot \text{Rank}(CB) + (1 - \alpha) \cdot \text{Rank}(CF)$$
- **Switching Hybrid:** Menangani *cold-start* secara adaptif:
  - $N=0$ rating: *Rule-Based (Popularitas)*
  - $1 \le N < 5$ rating: *Content-Based Filtering*
  - $N \ge 5$ rating: *Collaborative Filtering (SVD)*


In [ ]:
def switching_hybrid_recommender(user_ratings_dict, topn=5):
    n_ratings = len(user_ratings_dict)
    if n_ratings == 0:
        print("Strategi Aktif: Rule-Based (Popularitas) -> Mengatasi Cold-Start Total")
        return item_meta.sort_values("wr", ascending=False).head(topn)[["title", "author", "wr"]]
    elif n_ratings < 5:
        print(f"Strategi Aktif: Content-Based Filtering (Riwayat: {n_ratings} buku < Ambang 5)")
        return predict_content_based(user_ratings_dict, topn=topn)
    else:
        print(f"Strategi Aktif: Collaborative Filtering SVD (Riwayat: {n_ratings} buku >= Ambang 5)")
        return predict_svd(user_ratings_dict, topn=topn)

print("Uji Coba Transisi Cold-Start:")
print("1. Pengguna tanpa riwayat:")
display(switching_hybrid_recommender({}, topn=3))
print("\n2. Pengguna dengan riwayat cukup:")
display(switching_hybrid_recommender(sample_profile, topn=3))


---
## Bagian 6: Evaluasi Empiris Model (Hold-Out Benchmark)

Untuk mengevaluasi model secara saintifik, dilakukan pengujian *offline hold-out*:
1. Mengambil sampel $N=200$ pengguna yang memiliki minimal 10 rating.
2. Menyembunyikan 20% rating tiap pengguna sebagai **Test Set**.
3. Model dilatih hanya pada 80% sisa rating (**Training Set**).
4. Diukur 2 jenis performa:
   - **Akurasi Prediksi Rating:** Root Mean Squared Error (RMSE) dan Mean Absolute Error (MAE).
   - **Kualitas Perankingan Top-K ($K=10$):** Precision@K, Recall@K, dan HitRate@K (relevan jika rating asli $\ge 7$).


In [ ]:
def run_offline_evaluation(n_sample_users=150, test_frac=0.2, k=10, seed=42):
    rng = np.random.default_rng(seed)
    cand_users = np.where(np.diff(R.indptr) >= 10)[0]
    selected_users = rng.choice(cand_users, size=min(n_sample_users, len(cand_users)), replace=False)
    
    # Pisahkan test set
    test_data = {}
    for u in selected_users:
        cols = R.indices[R.indptr[u]:R.indptr[u+1]]
        nt = max(1, int(round(len(cols) * test_frac)))
        tc = rng.choice(cols, size=nt, replace=False)
        test_data[u] = (tc, np.asarray(R[u, tc].todense()).ravel())
    
    # Bangun matriks training R_tr
    coo = R.tocoo()
    n_items = R.shape[1]
    keys = coo.row.astype(np.int64) * n_items + coo.col
    test_keys = np.concatenate([u * n_items + test_data[u][0].astype(np.int64) for u in selected_users])
    keep = ~np.isin(keys, test_keys)
    R_tr = sparse.csr_matrix((coo.data[keep], (coo.row[keep], coo.col[keep])), shape=R.shape)
    
    # Latih model SVD pada data training
    cnt_tr = np.diff(R_tr.indptr)
    gm_tr = float(R_tr.data.mean())
    umean_tr = (np.asarray(R_tr.sum(1)).ravel() + SHRINK_MEAN * gm_tr) / (cnt_tr + SHRINK_MEAN)
    coo_tr = R_tr.tocoo()
    Rc_tr = sparse.csr_matrix((coo_tr.data - umean_tr[coo_tr.row], (coo_tr.row, coo_tr.col)), shape=R_tr.shape)
    svd_tr = TruncatedSVD(n_components=50, random_state=42).fit(Rc_tr)
    Vt_tr = svd_tr.components_.astype(np.float32)
    V_tr = Vt_tr.T.copy()
    
    # Evaluasi metrik untuk SVD vs Popularitas
    err_svd, err_pop = [], []
    hits_svd, hits_pop = [], []
    
    for u in selected_users:
        s, e = R_tr.indptr[u], R_tr.indptr[u+1]
        u_idx, u_r = R_tr.indices[s:e], R_tr.data[s:e]
        tc, tr = test_data[u]
        
        # Prediksi SVD
        mu_u = float((u_r.sum() + SHRINK_MEAN * gm_tr) / (len(u_r) + SHRINK_MEAN))
        v_u = sparse.csr_matrix((u_r - mu_u, (np.zeros(len(u_idx), dtype=int), u_idx)), shape=(1, n_items))
        z = v_u @ V_tr
        pred_svd_all = mu_u + np.asarray(z @ Vt_tr).ravel()
        
        # Prediksi Popularitas
        pop_pred = np.full(len(tc), gm_tr)
        svd_pred = np.clip(pred_svd_all[tc], 1, 10)
        
        err_svd.extend((svd_pred - tr).tolist())
        err_pop.extend((pop_pred - tr).tolist())
        
        # Ranking HitRate@K
        rel_books = set(tc[tr >= LIKE_THR])
        if rel_books:
            pred_svd_all[u_idx] = -np.inf
            top_k_svd = np.argpartition(-pred_svd_all, k-1)[:k]
            hits_svd.append(1.0 if len(rel_books.intersection(top_k_svd)) > 0 else 0.0)
            
            top_k_pop = np.argpartition(-item_meta["wr"].values, k-1)[:k]
            hits_pop.append(1.0 if len(rel_books.intersection(top_k_pop)) > 0 else 0.0)

    results = pd.DataFrame([
        {
            "Model": "Baseline (Popularitas Global)",
            "RMSE": np.sqrt(np.mean(np.array(err_pop)**2)),
            "MAE": np.mean(np.abs(err_pop)),
            f"HitRate@{k}": np.mean(hits_pop)
        },
        {
            "Model": "Collaborative Filtering (SVD)",
            "RMSE": np.sqrt(np.mean(np.array(err_svd)**2)),
            "MAE": np.mean(np.abs(err_svd)),
            f"HitRate@{k}": np.mean(hits_svd)
        }
    ])
    return results

print("Menjalankan Evaluasi Hold-out Benchmark...")
eval_df = run_offline_evaluation(n_sample_users=150, k=10)
display(eval_df)


---
## Bagian 7: Kesimpulan & Implementasi ke Dashboard Streamlit

### Temuan Utama:
1. **Pentingnya Normalisasi:** Penggunaan *Bayesian Shrinkage* dan *Mean-Centering* terbukti krusial untuk menstabilkan prediksi pada pengguna dengan riwayat rating yang minim.
2. **Kelebihan Collaborative Filtering (SVD):** SVD menghasilkan *RMSE* dan *HitRate@K* yang lebih unggul dibandingkan baseline popularitas karena mampu menangkap faktor laten preferensi pembaca.
3. **Pentingnya Hybrid Strategy:** Karena lebih dari 50% pengguna berada dalam status *cold-start*, sistem produksi harus mengintegrasikan *Rule-Based* dan *Content-Based* sebagai jembatan sebelum beralih ke *Collaborative Filtering*.

### Integrasi ke Dashboard Web:
Seluruh fungsi komputasi matriks dan pemodelan pada notebook ini dienkapsulasi ke dalam modul `recsys.py` dan disajikan dalam antarmuka interaktif multi-halaman pada `app.py` menggunakan framework **Streamlit** bertema **Donezo (Forest Green & Mint)**.
